# 05 Anomaly Detection

This notebook identifies outliers in the car auction dataset using two unsupervised methods: **Isolation Forest** and **Local Outlier Factor (LOF)**. The goal is to flag data points that deviate significantly from normal patterns -- these may represent data entry errors, unusual market transactions, or genuinely rare vehicles.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler

sns.set_style('darkgrid')

In [2]:
car_df = pd.read_csv('data/raw/car_prices.csv')
car_df = car_df.dropna(subset=['condition', 'odometer', 'sellingprice'])
print(f"Dataset: {car_df.shape[0]:,} rows")
car_df.head()

Dataset: 546,988 rows


,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,mmr,sellingprice,saledate
0,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg566472,ca,5.0,16639.0,white,black,"kia motors america, inc",20500,21500,Tue Dec 16 2014 12:30:00 GMT-0800 (PST)
1,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg561319,ca,5.0,9393.0,white,beige,"kia motors america, inc",20800,21500,Tue Dec 16 2014 12:30:00 GMT-0800 (PST)
2,2014,BMW,3 Series,328i SULEV,Sedan,automatic,wba3c1c51ek116351,ca,4.5,1331.0,gray,black,financial services remarketing (lease),31900,30000,Thu Jan 15 2015 04:30:00 GMT-0800 (PST)
3,2015,Volvo,S60,T5,Sedan,automatic,yv1612tb4f1310987,ca,4.1,14282.0,white,black,volvo na rep/world omni,27500,27750,Thu Jan 29 2015 04:30:00 GMT-0800 (PST)
4,2014,BMW,6 Series Gran Coupe,650i,Sedan,automatic,wba6b2c57ed129731,ca,4.3,2641.0,gray,black,financial services remarketing (lease),66000,67000,Thu Dec 18 2014 12:30:00 GMT-0800 (PST)


## Feature Selection and Scaling

We focus anomaly detection on three features that together describe a vehicle's expected value: condition rating, mileage, and selling price. Scaling ensures that no single feature dominates the distance calculations.

In [3]:
features = car_df[['condition', 'odometer', 'sellingprice']]

scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

## Isolation Forest

Isolation Forest works by randomly partitioning the feature space. Anomalies are "isolated" faster (in fewer splits) because they sit far from the dense regions of normal data. A contamination rate of 0.1% keeps the threshold conservative.

In [4]:
iforest = IsolationForest(n_estimators=1000, contamination=0.001, random_state=42)
iforest_labels = iforest.fit_predict(features_scaled)

car_df['iforest_label'] = iforest_labels
iforest_anomalies = car_df[car_df['iforest_label'] == -1]

print(f"Isolation Forest anomalies: {len(iforest_anomalies):,} ({len(iforest_anomalies)/len(car_df)*100:.2f}%)")
print(f"Normal data points:         {(car_df['iforest_label'] == 1).sum():,}")
print(f"\nSample anomalies:")
iforest_anomalies[['condition', 'odometer', 'sellingprice', 'make', 'model']].head(10)

Isolation Forest anomalies: 547 (0.10%)
Normal data points:         546,441

Sample anomalies:


,condition,odometer,sellingprice,make,model
275,1.0,999999.0,2500,Hyundai,Elantra Coupe
2470,1.0,291087.0,3600,Toyota,Corolla
4941,1.0,311164.0,700,Ford,F-250 Super Duty
5213,1.0,288484.0,800,BMW,7 Series
5272,1.0,287704.0,400,Saturn,S-Series
5318,5.0,5357.0,73000,Jaguar,F-TYPE
5357,4.9,183.0,75000,Jaguar,F-TYPE
5379,4.9,4225.0,83500,Jaguar,XJ
5570,2.0,10179.0,83500,Land Rover,Range Rover
5646,4.6,5369.0,135000,Land Rover,Range Rover


## Local Outlier Factor (LOF)

LOF measures how isolated a point is relative to its local neighborhood. Unlike Isolation Forest (which uses global partitioning), LOF can detect anomalies in datasets with varying densities. A point is flagged if its local density is significantly lower than that of its neighbors.

In [5]:
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.001)
lof_labels = lof.fit_predict(features_scaled)

car_df['lof_label'] = lof_labels
lof_anomalies = car_df[car_df['lof_label'] == -1]

print(f"LOF anomalies: {len(lof_anomalies):,} ({len(lof_anomalies)/len(car_df)*100:.2f}%)")
print(f"Normal data points: {(car_df['lof_label'] == 1).sum():,}")
print(f"\nSample anomalies:")
lof_anomalies[['condition', 'odometer', 'sellingprice', 'make', 'model']].head(10)

LOF anomalies: 547 (0.10%)
Normal data points: 546,441

Sample anomalies:


/Users/roy/miniconda3/envs/car_analysis_env/lib/python3.11/site-packages/sklearn/neighbors/_lof.py:322: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(


,condition,odometer,sellingprice,make,model
18,1.7,13441.0,17000,Chevrolet,Camaro
2998,1.8,88389.0,6800,Ford,Explorer Sport Trac
4135,1.8,133727.0,1350,Kia,Sedona
4139,1.7,87958.0,14700,Dodge,Ram Pickup 1500
4252,1.8,119294.0,3500,Ford,Mustang
4287,1.8,205256.0,3300,Toyota,Prius
4603,1.8,197843.0,3300,Toyota,Sequoia
5113,2.0,1.0,200,Buick,Regal
7565,2.0,1.0,250,Kia,Sedona
8054,1.0,154633.0,19600,Ram,3500


## Comparison: Overlap Between Methods

In [6]:
both_anomalous = car_df[(car_df['iforest_label'] == -1) & (car_df['lof_label'] == -1)]
print(f"Flagged by both methods: {len(both_anomalous):,}")
print(f"Flagged by Isolation Forest only: {((car_df['iforest_label'] == -1) & (car_df['lof_label'] == 1)).sum():,}")
print(f"Flagged by LOF only: {((car_df['iforest_label'] == 1) & (car_df['lof_label'] == -1)).sum():,}")

Flagged by both methods: 6
Flagged by Isolation Forest only: 541
Flagged by LOF only: 541


## Investigating Notable Anomalies

In [7]:
car_df['price_to_mmr'] = car_df['sellingprice'] / car_df['mmr']
avg_ratio = car_df['price_to_mmr'].mean()
print(f"Average price-to-MMR ratio (full dataset): {avg_ratio:.2f}")

if len(both_anomalous) > 0:
    print(f"\nPrice-to-MMR ratio for vehicles flagged by both methods:")
    anomaly_ratios = car_df.loc[both_anomalous.index, 'price_to_mmr'].describe()
    print(anomaly_ratios.round(2))

Average price-to-MMR ratio (full dataset): 0.99

Price-to-MMR ratio for vehicles flagged by both methods:
count     6.00
mean      2.89
std       4.37
min       0.47
25%       0.96
50%       1.19
75%       1.69
max      11.76
Name: price_to_mmr, dtype: float64


## Conclusion

Both Isolation Forest and LOF identify a small but meaningful set of outliers (~0.1% of the dataset). The detected anomalies fall into two main categories:

1. **Likely data errors** -- vehicles with placeholder values like 999,999 miles or suspiciously low condition ratings paired with high prices.
2. **Genuine market outliers** -- luxury vehicles with very low mileage and high prices, or high-mileage vehicles that sold for unexpectedly high amounts.

The two methods flag different (but overlapping) subsets, which is expected since they use fundamentally different approaches: Isolation Forest relies on random partitioning depth, while LOF compares local densities. Using both provides a more robust picture of anomalous patterns in the data.